# 01. 데이터 탐색
수집된 PubMed 논문 데이터를 분석합니다.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import json
from collections import Counter
import matplotlib.pyplot as plt
from src.ingestion.pubmed_fetcher import search_pubmed, fetch_paper_details
from src.ingestion.preprocessor import preprocess_papers
from src.ingestion.text_chunker import chunk_papers
from dotenv import load_dotenv
load_dotenv("../.env")

## 논문 수집

In [ ]:
pmids = search_pubmed("diabetes treatment", max_results=50)
papers = fetch_paper_details(pmids)
papers = preprocess_papers(papers)
print(f"수집된 논문 수: {len(papers)}")

## 데이터프레임 변환

In [ ]:
df = pd.DataFrame(papers)
df.head()

## 연도별 분포

In [ ]:
year_counts = df["year"].value_counts().sort_index()
plt.figure(figsize=(10,4))
plt.bar(year_counts.index, year_counts.values)
plt.title("논문 연도별 분포")
plt.xlabel("연도")
plt.ylabel("논문 수")
plt.tight_layout()
plt.show()

## 초록 길이 분포

In [ ]:
df["abstract_len"] = df["abstract"].apply(len)
df["abstract_len"].describe()

## 청킹 결과 확인

In [ ]:
chunks = chunk_papers(papers)
print(f"총 청크 수: {len(chunks)}")
print(f"논문당 평균 청크: {len(chunks)/len(papers):.1f}")
chunk_lengths = [len(c["text"]) for c in chunks]
print(f"청크 평균 길이: {sum(chunk_lengths)/len(chunk_lengths):.0f} 글자")

## 데이터 저장

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_json("../data/processed/papers.json", orient="records", force_ascii=False, indent=2)
print("저장 완료: data/processed/papers.json")